In [1]:
import sys
sys.path.append('../src')

import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

from model import SimpleCNN
from partition import dirichlet_partition
from federated import local_train
from defense import apply_dp_flat, apply_dp_adaptive, compute_skew_score

transform = transforms.ToTensor()
train_dataset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)
train_labels = [label for _, label in train_dataset]

num_clients = 5
alpha = 0.5
client_data = dirichlet_partition(train_labels, num_clients=num_clients, alpha=alpha)

# global label distribution (needed for skew score)
global_counts = np.array([np.sum(np.array(train_labels) == c) for c in range(10)])
global_dist = global_counts / global_counts.sum()

model = SimpleCNN(num_classes=10)

for client_id in range(num_clients):
    subset = Subset(train_dataset, client_data[client_id])
    dataloader = DataLoader(subset, batch_size=32, shuffle=True)
    
    state_dict = local_train(model, dataloader, epochs=1, lr=0.01)
    
    client_label_subset = [train_labels[i] for i in client_data[client_id]]
    skew = compute_skew_score(client_label_subset, global_dist)
    
    flat_dp_state = apply_dp_flat(state_dict, clip_norm=1.0, noise_scale=0.01)
    adaptive_dp_state = apply_dp_adaptive(state_dict, skew_score=skew, base_clip=1.0, base_noise=0.01)
    
    print(f"Client {client_id}: skew score = {skew:.4f}")

Client 0: skew score = 0.6740
Client 1: skew score = 0.5430
Client 2: skew score = 0.3352
Client 3: skew score = 0.3648
Client 4: skew score = 0.7428
